<a href="https://colab.research.google.com/github/vladislava-c/process-automation-analyzer/blob/main/Process_to_automation_ideas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai gradio -q

In [ ]:
# Each step = (step text, your label, set of acceptable Microsoft tools, or {"-"} for MANUAL)
# Borderline tools are now part of the set, so they count as correct.
TOOLS = ["Power Automate", "Power Apps", "Copilot", "SharePoint", "Outlook", "Teams", "Microsoft Forms", "Power BI"]

LABELS = ["AUTOMATE", "ASSIST", "MANUAL"]

ANSWER_KEY = [
 {"id": 16, "process": "A customer applies for a small business loan online. We check the documents, an analyst writes a recommendation, the credit committee decides, and the customer is told the outcome.",
  "steps": [
   ("Collect loan application and documents online", "AUTOMATE", {"Microsoft Forms", "Power Apps"}),
   ("Check that all required documents are present", "AUTOMATE", {"Power Automate"}),
   ("Draft credit recommendation for the analyst", "ASSIST", {"Copilot"}),
   ("Credit committee decides on the loan", "MANUAL", {"-"}),
   ("Inform customer of the decision", "AUTOMATE", {"Outlook"})]},                      # borderline label: rejections may need ASSIST

 {"id": 17, "process": "Every quarter the system owners must confirm who still needs access. Someone exports the user list, sends it to each owner, the owners confirm who can stay, unusual access is looked at by security, and unconfirmed accounts are removed. Completion rates are reported.",
  "steps": [
   ("Export current user access list", "AUTOMATE", {"Power Automate"}),
   ("Send list to each system owner for review", "AUTOMATE", {"Outlook", "Power Automate", "Teams"}),
   ("Owner confirms who keeps access", "MANUAL", {"-"}),
   ("Flag unusual or high-risk access", "ASSIST", {"Copilot"}),
   ("Remove accounts that were not confirmed", "AUTOMATE", {"Power Automate"}),       # borderline label: MANUAL in many companies
   ("Report review completion rates", "AUTOMATE", {"Power BI"})]},

 {"id": 18, "process": "A customer asks for a copy of their personal data. Compliance verifies who they are, collects the data from the different systems, checks that it contains nobody else's information, approves the release, and makes sure the 30-day deadline is met.",
  "steps": [
   ("Receive and log the request", "AUTOMATE", {"Microsoft Forms", "Outlook"}),
   ("Verify the identity of the requester", "MANUAL", {"-"}),
   ("Collect data from the different systems", "MANUAL", {"-"}),                       # borderline label: AUTOMATE if connectors exist
   ("Check the data for other people's information", "ASSIST", {"Copilot"}),
   ("Approve release of the data package", "MANUAL", {"-"}),
   ("Track the 30-day deadline", "AUTOMATE", {"Power Automate", "SharePoint"})]},

 {"id": 19, "process": "so basically when someone from sales wants a discount over 10% they message me, i look at the customer history, ask my boss if its ok, then tell sales yes or no and put it in the excel",
  "steps": [
   ("Receive discount request from sales", "AUTOMATE", {"Microsoft Forms", "Teams"}),
   ("Review the customer's history", "ASSIST", {"Copilot"}),
   ("Route the request to the manager for approval", "AUTOMATE", {"Power Automate"}),
   ("Manager decides on the discount", "MANUAL", {"-"}),
   ("Tell sales the decision", "AUTOMATE", {"Teams"}),
   ("Record the decision in the spreadsheet", "AUTOMATE", {"SharePoint", "Power Automate"})]},

 {"id": 20, "process": "we get supplier emails all the time with price changes. somebody reads them, updates our price list, and if its a big jump they call the supplier. once a month the buyers meet and talk about which suppliers to push on price",
  "steps": [
   ("Read supplier emails about price changes", "ASSIST", {"Copilot"}),
   ("Update the price list", "AUTOMATE", {"Power Automate", "SharePoint"}),
   ("Flag price increases above a threshold", "AUTOMATE", {"Power Automate"}),
   ("Call the supplier about large increases", "MANUAL", {"-"}),
   ("Summarise price trends for the monthly buyer meeting", "ASSIST", {"Copilot", "Power BI"}),
   ("Decide which suppliers to push on price", "MANUAL", {"-"})]},
]

# Sanity check: every tool in the key must be a real tool name
for p in ANSWER_KEY:
    for step, label, tools in p["steps"]:
        assert label in LABELS, (p["id"], step)
        assert tools <= set(TOOLS) | {"-"}, (p["id"], step, tools)
        assert (label == "MANUAL") == (tools == {"-"}), (p["id"], step)
print("Loaded", len(ANSWER_KEY), "processes,", sum(len(p["steps"]) for p in ANSWER_KEY), "steps")

In [ ]:
from google.colab import userdata
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=userdata.get("API"))
MODEL = "gpt-4o-mini"      # try a newer small model too and compare the eval scores
TEMPERATURE = 0            # keeps runs comparable while you tune the prompt

# ---------- Stage 1: split a free-text process into steps ----------
SPLIT_INSTRUCTIONS = """
You break a business process description into its individual steps.
- 3 to 8 steps, in the order they happen.
- Each step is one short imperative phrase, e.g. "Check that all required documents are present".
- Keep every human decision (approve, decide, negotiate, sign off) as its own separate step.
- Do not invent steps that the description does not imply.
Also give a short title for the process (max 6 words).
"""

SPLIT_SCHEMA = {"type": "json_schema", "json_schema": {"name": "process_steps", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["process", "steps"],
    "properties": {
        "process": {"type": "string"},
        "steps": {"type": "array", "items": {"type": "string"}}}}}}

# ---------- Stage 2: label each step (this is what the eval measures) ----------
LABEL_INSTRUCTIONS = f"""
You analyze business processes for a company that uses the Microsoft landscape.
You get a process description and a numbered list of its steps. For each step:
1. Write a one-sentence reason.
2. Assign exactly one label:
   - AUTOMATE: rule-based, repeatable, structured data, no judgment needed.
   - ASSIST: AI or software prepares something (a draft, a summary, a classification, a comparison, a proposal) and a human reviews or approves it.
   - MANUAL: a decision, negotiation, investigation, relationship, accountability, or legal sign-off that a person must own.
3. Pick the best-fit tool, using exactly one name from: {", ".join(TOOLS)}. For MANUAL steps the tool is "-".
Return one entry per step, with n equal to the step number.

Labelling rules:
- Use ASSIST, not MANUAL, whenever AI could do the first pass of the work. This includes reading, reviewing,
  screening or summarising text such as emails, records, logs or customer history. The human decision that
  follows is a separate step.
- A check against a fixed rule or threshold is AUTOMATE. A judgment about what looks unusual, risky or relevant is ASSIST.
- Exception: we have no document-extraction tooling. Pulling structured fields (amounts, dates, names) out of
  attached or scanned files such as invoices, receipts, CVs or contracts into a system is MANUAL with tool "-".
  This applies ONLY to that extraction step. Steps that collect, route, store, send, or check the completeness
  of documents are still AUTOMATE.

Tool rules:
- Sending or scheduling email or calendar items: Outlook
- Notifying or messaging people in chat: Teams
- Collecting submissions or survey responses: Microsoft Forms
- Building a custom input screen or app for users: Power Apps
- Routing, approvals, data transfer, triggers, and other system-to-system steps: Power Automate
- Storing, tracking, or publishing documents and lists: SharePoint
- Dashboards and reports: Power BI
- Summarising, drafting, classifying, comparing, or reading text: Copilot

Worked examples (not from the real data):
Process: Employees submit expense claims with receipts, finance checks them against policy, the manager approves, and the claim is paid.
1. Submit expense claim with receipts -> AUTOMATE, Microsoft Forms
2. Copy amounts and dates from the receipts into the finance system -> MANUAL, - (field extraction from files)
3. Check the claim against the expense policy -> ASSIST, Copilot (AI compares, finance confirms)
4. Manager approves the claim -> MANUAL, -
5. Send the approved claim to payroll -> AUTOMATE, Power Automate

Process: Customer complaints arrive by email; every week the team reviews them and decides which to escalate.
1. Summarise this week's complaint emails -> ASSIST, Copilot
2. Decide which complaints to escalate -> MANUAL, -
3. Remind the team of the weekly review -> AUTOMATE, Teams
"""

LABEL_SCHEMA = {"type": "json_schema", "json_schema": {"name": "step_labels", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["steps"],
    "properties": {"steps": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "required": ["n", "reason", "label", "tool"],          # reason first: the model explains before it labels
        "properties": {
            "n": {"type": "integer"},
            "reason": {"type": "string"},
            "label": {"type": "string", "enum": LABELS},
            "tool": {"type": "string", "enum": TOOLS + ["-"]}}}}}}}}


def _call(system, user, schema):
    r = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        response_format=schema,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    return json.loads(r.choices[0].message.content)


def split_steps(process_text):
    """Returns (title, [step, ...])."""
    data = _call(SPLIT_INSTRUCTIONS, process_text, SPLIT_SCHEMA)
    steps = [s.strip() for s in data["steps"] if s.strip()]
    return data["process"].strip(), steps


def label_steps(process_text, steps):
    """Returns one dict {reason, label, tool} per input step, in the same order (matched on n, not position)."""
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(steps))
    data = _call(LABEL_INSTRUCTIONS, f"Process: {process_text}\n\nSteps:\n{numbered}", LABEL_SCHEMA)
    by_n = {s["n"]: s for s in data["steps"]}
    out = []
    for i in range(1, len(steps) + 1):
        s = by_n.get(i, {"reason": "model returned no entry for this step", "label": "?", "tool": "?"})
        tool = "-" if s["label"] == "MANUAL" else s["tool"]
        out.append({"reason": s["reason"], "label": s["label"], "tool": tool})
    return out

print("Ready:", MODEL)

In [ ]:
N_RUNS = 3   # 29 steps is small: one flipped step moves the score ~3.5 points, so average over runs

rows = []
for run in range(N_RUNS):
    for p in ANSWER_KEY:
        steps = [s[0] for s in p["steps"]]
        try:
            out = label_steps(p["process"], steps)
        except Exception as e:
            print(f"Run {run}, process {p['id']}: {e}")
            continue
        for (step, key_label, key_tools), m in zip(p["steps"], out):
            rows.append({"run": run, "id": p["id"], "step": step,
                         "key_label": key_label, "model_label": m["label"],
                         "key_tools": " / ".join(sorted(key_tools)), "model_tool": m["tool"],
                         "tool_ok": m["tool"] in key_tools,
                         "has_tool": key_tools != {"-"},
                         "reason": m["reason"]})

df = pd.DataFrame(rows)
df["label_ok"] = df["key_label"] == df["model_label"]

tool_rows = df[df["has_tool"]]
tool_rows_label_ok = tool_rows[tool_rows["label_ok"]]   # tool score without double-counting label misses

per_run = df.groupby("run")["label_ok"].mean()
print(f"Label accuracy: {df['label_ok'].mean():.0%}  (per run: {', '.join(f'{x:.0%}' for x in per_run)})")
print(f"Tool accuracy, label correct:  {tool_rows_label_ok['tool_ok'].mean():.0%}  ({len(tool_rows_label_ok)} step-runs)")
print(f"Tool accuracy, all automatable: {tool_rows['tool_ok'].mean():.0%}  ({len(tool_rows)} step-runs)")
print()
print(pd.crosstab(df["key_label"], df["model_label"], rownames=["Your key"], colnames=["Model said"]))

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)

# Every miss, with how many runs it happened in and the model's reason
misses = df[~df["label_ok"] | (df["has_tool"] & df["label_ok"] & ~df["tool_ok"])]
(misses.groupby(["id", "step", "key_label", "model_label", "key_tools", "model_tool"])
       .agg(runs=("run", "count"), reason=("reason", "first"))
       .reset_index()
       .sort_values(["runs", "id"], ascending=[False, True]))

In [ ]:
import tempfile
import traceback
import gradio as gr

BADGE = {"AUTOMATE": "🟢 AUTOMATE", "ASSIST": "🟡 ASSIST", "MANUAL": "🔴 MANUAL"}


def do_split(text):
    if not text or not text.strip():
        raise gr.Error("Please describe a process first.")
    try:
        title, steps = split_steps(text)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Could not split the process: {e}") from e
    return title, "\n".join(steps)


def do_label(text, title, steps_text):
    # Uses the same label_steps() that the eval measures
    if not text or not text.strip():
        raise gr.Error("Please describe a process first.")
    steps = [s.strip() for s in (steps_text or "").splitlines() if s.strip()]
    if not steps:                                   # user skipped step 1: split automatically
        title, steps_text = do_split(text)
        steps = steps_text.splitlines()
    try:
        out = label_steps(text, steps)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Could not label the steps: {e}") from e

    result = pd.DataFrame({"Step": steps,
                           "Label": [m["label"] for m in out],
                           "Tool": [m["tool"] for m in out],
                           "Reason": [m["reason"] for m in out]})

    counts = result["Label"].value_counts()
    summary = (f"**{title or 'Process'}**: {len(result)} steps · "
               f"{counts.get('AUTOMATE', 0)} automate · {counts.get('ASSIST', 0)} assist · {counts.get('MANUAL', 0)} manual")

    csv_path = tempfile.NamedTemporaryFile(delete=False, suffix=".csv", prefix="process_analysis_").name
    result.assign(Process=title)[["Process", "Step", "Label", "Tool", "Reason"]].to_csv(csv_path, index=False)

    shown = result.assign(Label=result["Label"].map(lambda l: BADGE.get(l, l)))
    return title, "\n".join(steps), shown, summary, csv_path


example = ("A client submits a form to open an account. Staff check that all fields are filled in, "
           "enter the data in the core system, review sanctions screening hits, and send a welcome email.")

with gr.Blocks(title="Process Automation Analyzer") as demo:
    gr.Markdown("# Process Automation Analyzer\nDescribe a process. It is split into steps, and each step is labelled "
                "AUTOMATE / ASSIST / MANUAL with a suggested Microsoft tool.")
    box = gr.Textbox(lines=6, label="1. Describe the process", placeholder="Describe your process in plain words...")
    gr.Examples([[example]], inputs=box)
    split_btn = gr.Button("Split into steps")
    with gr.Row():
        title_box = gr.Textbox(label="Title", scale=1)
        steps_box = gr.Textbox(lines=8, label="2. Steps (one per line, edit freely)", scale=3)
    label_btn = gr.Button("Label steps", variant="primary")
    summary_md = gr.Markdown()
    out = gr.Dataframe(label="Result", wrap=True)
    csv_file = gr.File(label="Download CSV")

    split_btn.click(do_split, inputs=box, outputs=[title_box, steps_box])
    label_btn.click(do_label, inputs=[box, title_box, steps_box],
                    outputs=[title_box, steps_box, out, summary_md, csv_file])
    box.change(lambda: ("", ""), outputs=[title_box, steps_box])   # new process text -> clear stale steps

demo.launch(debug=True)